In [2]:
!wget https://www.dropbox.com/s/pdhwlpi2yeie0ol/movie-reviews-dataset.zip

--2026-07-29 01:31:51--  https://www.dropbox.com/s/pdhwlpi2yeie0ol/movie-reviews-dataset.zip
Resolving www.dropbox.com (www.dropbox.com)... 162.125.1.18, 2620:100:6016:18::a27d:112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.1.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/4r8fb499vfpyrw44fgftj/movie-reviews-dataset.zip?rlkey=79qfzf6683udd2ehdii38y7wt [following]
--2026-07-29 01:31:51--  https://www.dropbox.com/scl/fi/4r8fb499vfpyrw44fgftj/movie-reviews-dataset.zip?rlkey=79qfzf6683udd2ehdii38y7wt
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uce7fb17822eeda708735cc9db1d.dl.dropboxusercontent.com/cd/0/inline/DFIdMUXNoRVeXJv7MZ297vAar22H_sDkdHlgdRh_OTstB6nRxjP689pQSdtx0YHa5dqFhx2DNztbJkHqJhmolo0cQHhEt3Rc5pVaMPT9prnGTXsi_ViICuqKmgyF11yQTPsrl5tDZ0MK0X7N_BfmYG_0/file# [following]
--2026-07-29 01:31:51--  https://uce7fb17822eeda708735cc9db1d.

In [3]:
!unzip -q "/content/movie-reviews-dataset.zip"

In [4]:
from tensorflow.keras.preprocessing import text_dataset_from_directory
from tensorflow.strings import regex_replace
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.models import Sequential
from tensorflow.keras import Input
from tensorflow.keras.layers import Dense, RNN, SimpleRNNCell, Embedding, Dropout

In [5]:
def prepareData(dir):
  data = text_dataset_from_directory(dir)
  return data.map(
    lambda text, label: (regex_replace(text, '<br />', ' '), label),
  )

In [6]:
train_data = prepareData('movie-reviews-dataset/train')
test_data = prepareData('movie-reviews-dataset/test')

# print a sample text from the traindataset
for text_batch, label_batch in train_data.take(1):
  print(text_batch.numpy()[0])
  print(label_batch.numpy()[0])

Found 25000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.
b'There have been plenty of unknown movies or movies given bad reviews that I really liked. This was not one of them.  It was overacted and used camera techniques that made me feel like I was watching a soap opera. It was ludicrously predictable and took most of the movie to get going then left you asking "that\'s it?". Once I decided not to take the movie too seriously and watch it from a purely corny point of view it became more enjoyable. This is one movie that would have wound up on MST3000 if it was still on.'
0


In [7]:
model = Sequential()

In [8]:
model.add(Input(shape=(), dtype="string"))

In [9]:
max_tokens = 1000
max_len = 100
vectorize_layer = TextVectorization(
  max_tokens=max_tokens,
  output_mode="int",
  output_sequence_length=max_len,
)

In [10]:
train_texts = train_data.map(lambda text, label: text)
vectorize_layer.adapt(train_texts)

model.add(vectorize_layer)

In [11]:
model.add(Embedding(max_tokens + 1, 128))

rnn = RNN(SimpleRNNCell(64) , return_sequences=False,return_state=False)
model.add(rnn)
model.add(Dense(64, activation="relu"))
model.add(Dense(1, activation="sigmoid"))

In [12]:
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [13]:
model.fit(train_data, epochs=10)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 94s 115ms/step - accuracy: 0.4994 - loss: 0.6989
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 86s 110ms/step - accuracy: 0.5130 - loss: 0.6959
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 86s 111ms/step - accuracy: 0.5291 - loss: 0.6914
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 89s 113ms/step - accuracy: 0.5839 - loss: 0.6735
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 86s 110ms/step - accuracy: 0.6011 - loss: 0.6601
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 86s 110ms/step - accuracy: 0.5885 - loss: 0.6655
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 87s 111ms/step - accuracy: 0.5705 - loss: 0.6725
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 88s 112ms/step - accuracy: 0.6219 - loss: 0.6470
Epoch 9/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 85s 109ms/step - accuracy: 0.6016 - loss: 0.6596
Epoch 10/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 88s 113ms/step - accuracy: 0.5902 - loss: 0.6645


In [14]:
model.evaluate(test_data)

782/782 ━━━━━━━━━━━━━━━━━━━━ 37s 46ms/step - accuracy: 0.5406 - loss: 0.6933


[0.6933279633522034, 0.5405600070953369]

In [18]:
text = "I LOVE the movie !"

In [19]:
import tensorflow as tf
model.predict(tf.constant([text]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step


array([[0.6032248]], dtype=float32)